# Predictive Churn Analysis
Logistic Regression classification pipeline for customer retention risk.

The uploaded sample contains 15 records, so evaluation metrics are directional and should not be generalized.

In [ ]:
import pandas as pd
df = pd.read_csv('customer_churn_sample(1).csv')
df['ChurnTarget'] = (df['Churn']=='Yes').astype(int)
df.head()

## 1. Preprocessing
- Remove CustomerID from predictive features.
- One-hot encode categorical variables.
- Median-impute and MinMax-scale numeric variables.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X=df.drop(columns=['Churn','ChurnTarget','CustomerID']); y=df['ChurnTarget']
cat=X.select_dtypes(include='object').columns.tolist(); num=X.select_dtypes(exclude='object').columns.tolist()
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',MinMaxScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
model=Pipeline([('prep',prep),('clf',LogisticRegression(max_iter=2000,class_weight='balanced'))])
model.fit(X_train,y_train)
pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
print('Precision:',precision_score(y_test,pred,zero_division=0))
print('Recall:',recall_score(y_test,pred,zero_division=0))
print('F1:',f1_score(y_test,pred,zero_division=0))
print('ROC-AUC:',roc_auc_score(y_test,prob) if len(y_test.unique())==2 else 'undefined')

## 2. Risk Scoring
Refit the pipeline on all available records and export churn probability and risk level for every customer.

In [ ]:
model.fit(X,y)
df['ChurnProbability']=model.predict_proba(X)[:,1]
df['RiskLevel']=pd.cut(df['ChurnProbability'],bins=[-.001,.33,.66,1.001],labels=['Low','Medium','High'])
df[['CustomerID','Churn','ChurnProbability','RiskLevel']].sort_values('ChurnProbability',ascending=False).to_csv('customer_churn_risk_scores.csv',index=False)